# Oncologie en France — 4 jeux de données à explorer

Ce notebook construit **4 DataFrames** à partir des données ouvertes publiques (open data) de l'Assurance Maladie (ameli) et de la DREES.

**Démarche.** Chaque DataFrame est tiré d'**une seule source**, nettoyé, sans fusion. Les jointures possibles sont **signalées par 🔗** dans chaque section et récapitulées à la fin. Elles ne sont **pas réalisées**.

| # | DataFrame | Contenu | Source | Années | Géographie | Lignes |
|---|---|---|---|---|---|---:|
| 1 | `df_patients_sein` | Patients atteints d'un cancer du sein, par âge, sexe et territoire | ameli `effectifs` | 2015‑2024 | France, région, département | 151 200 |
| 2 | `df_depenses_onco` | Dépenses remboursées par pathologie oncologique et par poste | ameli `depenses` | 2015‑2024 | France | 4 960 |
| 3 | `df_dms_onco` | Durée moyenne de séjour (DMS) par pathologie tumorale et par territoire | DREES morbidité hospitalière | 2018‑2024 | France, région, département | 22 869 |
| 4 | `df_charge_onco` | Charge hospitalière : séjours par durée, part d'ambulatoire, journées, par pathologie tumorale et par territoire | DREES morbidité hospitalière | 2018‑2024 | France, région, département | 22 869 |

ℹ️ Les **dépenses par territoire** (ancien DF 5, obtenues en ventilant la dépense nationale au prorata des patients de chaque territoire) ont été **retirées** car peu fiables : la dépense par patient y était identique partout. Elles seront traitées plus tard.

*Nombre de lignes relevé le 03/11/2026. Il évoluera quand de nouvelles années seront publiées.*

### Conventions communes
- **`annee`** : entier (2015, 2016…).

#### Colonnes géographiques : `niveau`, `code_zone`, `code_region`, `dept`
Chaque ligne d'une table territoriale décrit **une zone** : la France entière, une région ou un département. Ces quatre colonnes indiquent laquelle.

| Colonne | Rôle | Valeurs possibles | Présente dans |
|---|---|---|---|
| **`niveau`** | Échelle géographique de la ligne | `France`, `Région`, `Département` | DF 1, 3, 4 |
| **`code_zone`** | **Code unique de la zone décrite par la ligne**, quel que soit le niveau. C'est la colonne à utiliser pour filtrer, grouper ou joindre. | `FR` = France entière ; `FRM` = France métropolitaine (DREES uniquement) ; sinon code INSEE de la région (`11`, `84`… ; DOM `01` à `06`) ou du département (`01`…`95`, `2A`, `2B`, `971`…`976`) | DF 1, 3, 4 |
| **`code_region`** | Code ameli brut de la **région à laquelle appartient** la ligne | `99` = France entière ; sinon code INSEE de la région | DF 1 (source ameli) |
| **`dept`** | Code ameli brut du **département** de la ligne | `999` = pas de département (ligne France ou total d'une région) ; sinon code du département | DF 1 |

**Comment `code_zone` est construit à partir des codes ameli** : `niveau = France` → `FR` ; `niveau = Région` → `code_region` ; `niveau = Département` → `dept`.

| `niveau` | `code_zone` | `code_region` | `dept` | Lecture |
|---|---|---|---|---|
| France | `FR` | `99` | `999` | France entière |
| Région | `11` | `11` | `999` | Total de l'Île‑de‑France |
| Département | `75` | `11` | `75` | Paris, qui appartient à l'Île‑de‑France |
| Département | `971` | `01` | `971` | Guadeloupe (dans les DOM, la région et le département se confondent) |

**Usages**
- Comparer les départements : `df[df['niveau'] == 'Département']`, puis grouper ou trier sur `code_zone`.
- Départements d'une région : `df[(df['niveau'] == 'Département') & (df['code_region'] == '11')]`.
- Les tables DREES (DF 3, DF 4) n'ont pas `code_region` mais ont `libelle_zone` (nom de la zone). 🔗 Pour retrouver la région d'un département DREES, joindre avec le DF 1 sur `code_zone`.
- Les codes de zone sont **identiques entre ameli et DREES**, ce qui rend les jointures géographiques directes.

⚠️ **France, régions et départements cohabitent dans chaque table territoriale** : la ligne France est déjà la somme des régions, et chaque région la somme de ses départements. Toujours filtrer sur `niveau` avant un `sum()` ou un `describe()`, sinon les mêmes patients ou séjours sont comptés trois fois.

In [29]:
import requests
import json
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

AMELI_URL = 'https://data.ameli.fr/api/explore/v2.1/catalog/datasets/{dataset}/exports/json'
RAW_DIR = Path('out/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)


def export_ameli(dataset, where):
    # Récupère tous les enregistrements d'un dataset ameli (endpoint d'export), filtrés en ODSQL
    response = requests.get(AMELI_URL.format(dataset=dataset),
                            headers={'Accept': 'application/json'},
                            params={'where': where})
    if response.status_code == 200:
        records = response.json()
        print(f"{dataset} : {len(records)} enregistrements récupérés")
        return pd.DataFrame(records)
    print(f"Erreur HTTP {response.status_code} : {response.text}")
    return pd.DataFrame()


def niveau_geo_ameli(df):
    # Codification ameli : region = '99' -> France ; dept = '999' -> total de la région ; sinon département
    df['niveau'] = np.select([df['region'] == '99', df['dept'] == '999'],
                             ['France', 'Région'], default='Département')
    df['code_zone'] = np.select([df['niveau'] == 'France', df['niveau'] == 'Région'],
                                ['FR', df['region']], default=df['dept'])
    return df

## DF 1 — `df_patients_sein` : patients atteints d'un cancer du sein, par âge, sexe et territoire

**Source** : ameli `effectifs` (cartographie des pathologies), filtre `patho_niv2 = "Cancer du sein de la femme"`.

**Démarche**
1. Extraction complète via l'endpoint d'export (aucune limite de 10 000 lignes).
2. `annee` convertie en entier ; `patho_niv3` vide remplacé par `Ensemble` (= actif + sous surveillance).
3. Ajout de `niveau` / `code_zone` à partir des codes ameli `region` / `dept`.
4. Colonnes renommées : `ntop` → `nb_patients`, `npop` → `population`, `prev` → `prevalence_pct` (en %).

**Une ligne** = année × sous‑pathologie (`Ensemble` / `actif` / `sous surveillance`) × classe d'âge × sexe × territoire.

⚠️ **Lignes de total mélangées aux lignes détaillées** :
| Dimension | Valeur « total » | Valeurs détaillées |
|---|---|---|
| `sexe` | `9` (tous sexes) | `2` (femmes) — pas de ligne hommes pour ce cancer |
| `cla_age_5` | `tsage` (tous âges) | `00-04` … `95et+` |
| `patho_niv3` | `Ensemble` | `… actif`, `… sous surveillance` |
| territoire | `niveau = France` puis `Région` | `Département` |

👉 Pour un total, on fixe **une seule valeur par dimension** (voir l'exemple sous le code).

⚠️ **`nb_patients` vide sur 47 536 lignes** : petits effectifs non diffusés (secret statistique). Ces cases ne valent pas zéro. Les effectifs diffusés sont arrondis à la dizaine.

🔗 **Jointures possibles (non réalisées)**
- avec **DF 3 / DF 4** sur (`annee`, `niveau`, `code_zone`) en filtrant `code_patho = '02017'` (tumeur maligne du sein) : DMS ou séjours rapportés aux patients, en prenant `cla_age_5 = 'tsage'`, `sexe = '9'` et `patho_niv3 = 'Ensemble'` ;
- avec le référentiel ameli `referentiel-departements` sur `dept` : libellés des départements.

In [30]:
df_patients_sein = export_ameli('effectifs', 'patho_niv2 = "Cancer du sein de la femme"')

df_patients_sein['annee'] = pd.to_datetime(df_patients_sein['annee']).dt.year
df_patients_sein['patho_niv3'] = df_patients_sein['patho_niv3'].fillna('Ensemble')
df_patients_sein = niveau_geo_ameli(df_patients_sein)

df_patients_sein = (df_patients_sein
    .rename(columns={'region': 'code_region', 'ntop': 'nb_patients', 'npop': 'population', 'prev': 'prevalence_pct'})
    [['annee', 'niveau', 'code_zone', 'code_region', 'dept', 'patho_niv2', 'patho_niv3', 'top',
      'cla_age_5', 'libelle_classe_age', 'sexe', 'libelle_sexe', 'nb_patients', 'population', 'prevalence_pct']]
    .sort_values(['annee', 'niveau', 'code_zone', 'top', 'sexe', 'cla_age_5'])
    .reset_index(drop=True))

print(f"df_patients_sein : {df_patients_sein.shape[0]} lignes x {df_patients_sein.shape[1]} colonnes")

effectifs : 151200 enregistrements récupérés
df_patients_sein : 151200 lignes x 15 colonnes


In [31]:
# Exemple : total France, tous âges, tous sexes, actif + sous surveillance
display(df_patients_sein[(df_patients_sein['niveau'] == 'France')
                         & (df_patients_sein['cla_age_5'] == 'tsage')
                         & (df_patients_sein['sexe'] == '9')
                         & (df_patients_sein['patho_niv3'] == 'Ensemble')]
        [['annee', 'nb_patients', 'population', 'prevalence_pct']])

,annee,nb_patients,population,prevalence_pct
12809,2015,685350.0,33622410,2.038
27929,2016,679610.0,33847120,2.008
43049,2017,695750.0,33988170,2.047
58169,2018,714910.0,34164050,2.093
73289,2019,727280.0,34379350,2.115
88409,2020,729310.0,34401380,2.120
103529,2021,731890.0,35242500,2.077
118649,2022,726650.0,35442250,2.050
133769,2023,727060.0,35047220,2.075
148889,2024,728980.0,35171580,2.073


## DF 2 — `df_depenses_onco` : dépenses remboursées par pathologie oncologique (France)

**Source** : ameli `depenses` (dépenses remboursées **affectées** à chaque pathologie), filtre `patho_niv1 = "Cancers"`.
Ce jeu de données n'existe **qu'au niveau national**.

**Démarche**
1. Extraction de tous les postes de dépense pour les cancers.
2. `patho_niv2` vide → `Tous cancers` ; `patho_niv3` vide → `Ensemble` (= actif + sous surveillance).
3. Ajout de **`type_depense`**, un regroupement lisible des 31 postes `dep_niv_2` (MCO séjours, liste en sus, actes externes, SSR, HAD, psychiatrie, ville par profession, prestations en espèces…), et de **`secteur`** (public / privé quand c'est précisé).
4. Colonnes renommées : `ntop` → `nb_patients_pathologie`, `montant` en euros.

**Une ligne** = année × pathologie (`top`) × poste de dépense.

**Pathologies disponibles** (`patho_niv2`) : Cancer du sein de la femme, Cancer colorectal, Cancer bronchopulmonaire, Cancer de la prostate, Autres cancers, Tous cancers. Chacune est déclinée en Ensemble / actif / sous surveillance.

⚠️ **Double comptage possible**
- `type_somme = 'Total'` : sous‑totaux (`TOTAL hôpital`, `TOTAL ville`, `TOTAL prestations en espèces`) et total général (`TOTAL général`). Pour additionner les postes, garder **`type_somme = 'Partiel'`**. Contrôle effectué : somme des postes « Partiel » = totaux, à 5 € près.
- `patho_niv3 = 'Ensemble'` = actif + sous surveillance. `Tous cancers` = somme des 5 localisations.

ℹ️ Les postes `Psychiatrie`, `Ville - sages-femmes` et `Prestations - IJ maternité et autres` existent mais valent **0 €** pour les cancers.

⚠️ Ce sont des montants **remboursés** par l'Assurance Maladie, affectés à la pathologie selon la méthode de la cartographie. Ce n'est ni un coût complet, ni le reste à charge des patients.

🔗 **Jointures possibles (non réalisées)**
- avec **DF 3 / DF 4**, au niveau national seulement (`code_zone = 'FR'`), sur `annee` et la **correspondance de pathologies** (tableau en fin de notebook) : par exemple dépense MCO et nombre de séjours ;
- avec **DF 1** (ou `effectifs`) sur (`annee`, `top`), au niveau France : `nb_patients_pathologie` est déjà identique au `nb_patients` France d'`effectifs`.

In [32]:
df_depenses_onco = export_ameli('depenses', 'patho_niv1 = "Cancers"')

df_depenses_onco['annee'] = pd.to_datetime(df_depenses_onco['annee']).dt.year
df_depenses_onco['patho_niv2'] = df_depenses_onco['patho_niv2'].fillna('Tous cancers')
df_depenses_onco['patho_niv3'] = df_depenses_onco['patho_niv3'].fillna('Ensemble')

# Regroupement lisible des postes de dépense (dep_niv_2)
TYPES_DEPENSE = {
    'Hospitalisations séjour MCO secteur public remboursées': 'Hôpital - séjours MCO',
    'Hospitalisations séjour MCO secteur privé remboursées': 'Hôpital - séjours MCO',
    'Hospitalisations liste en sus MCO secteur public remboursées': 'Hôpital - liste en sus MCO',
    'Hospitalisations liste en sus MCO secteur privé remboursées': 'Hôpital - liste en sus MCO',
    'Actes et consultations externes MCO secteur public remboursés': 'Hôpital - actes et consultations externes',
    'Hospitalisations en SSR secteur public remboursées': 'SSR (soins de suite et de réadaptation)',
    'Hospitalisations en SSR secteur privé remboursées': 'SSR (soins de suite et de réadaptation)',
    'Hospitalisations en HAD secteur public remboursées': 'HAD (hospitalisation à domicile)',
    'Hospitalisations en HAD secteur privé remboursées': 'HAD (hospitalisation à domicile)',
    'Hospitalisations en psychiatrie secteur public remboursées': 'Psychiatrie',
    'Hospitalisations en psychiatrie secteur privé remboursées': 'Psychiatrie',
    'Médicaments remboursés': 'Ville - médicaments',
    'Biologie remboursée': 'Ville - biologie',
    'Soins infirmiers remboursés': 'Ville - soins infirmiers',
    'Soins de kinésithérapie remboursés': 'Ville - kinésithérapie',
    "Soins d'autres paramédicaux remboursés": 'Ville - autres paramédicaux',
    'Soins de généralistes remboursés': 'Ville - généralistes',
    'Soins autres spécialistes remboursés': 'Ville - spécialistes',
    'Soins de sages-femmes remboursés': 'Ville - sages-femmes',
    'Soins dentaires remboursés': 'Ville - dentaire',
    'Transports remboursés': 'Ville - transports',
    'Autres produits de santé remboursés': 'Ville - autres produits de santé',
    'Autres dépenses de soins de ville remboursés': 'Ville - autres dépenses',
    'Indemnités journalières maladie et AT/MP remboursées': 'Prestations - IJ maladie et AT/MP',
    'Indemnités journalières maternité et autres prestations remboursées': 'Prestations - IJ maternité et autres',
    "Prestations d'invalidité remboursées": 'Prestations - invalidité',
    'Rentes de victimes ATMP remboursées': 'Prestations - rentes AT/MP',
    'Total hospitalisations (tous secteurs) remboursées': 'TOTAL hôpital',
    'Total soins de ville remboursés': 'TOTAL ville',
    'Total prestations en espèces remboursées': 'TOTAL prestations en espèces',
    'Total des dépenses remboursées': 'TOTAL général',
}
df_depenses_onco['type_depense'] = df_depenses_onco['dep_niv_2'].map(TYPES_DEPENSE)
df_depenses_onco['secteur'] = np.select(
    [df_depenses_onco['dep_niv_2'].str.contains('secteur public'),
     df_depenses_onco['dep_niv_2'].str.contains('secteur privé')],
    ['Public', 'Privé'], default='Non précisé')

df_depenses_onco = (df_depenses_onco
    .rename(columns={'ntop': 'nb_patients_pathologie'})
    [['annee', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1', 'type_depense', 'secteur', 'dep_niv_2',
      'type_somme', 'montant', 'nb_patients_pathologie', 'n_recourant_au_poste', 'montant_moy']]
    .sort_values(['annee', 'top', 'dep_niv_1', 'type_somme', 'dep_niv_2'])
    .reset_index(drop=True))

print(f"df_depenses_onco : {df_depenses_onco.shape[0]} lignes x {df_depenses_onco.shape[1]} colonnes")
print("Postes sans type :", df_depenses_onco['type_depense'].isna().sum())

depenses : 4960 enregistrements récupérés
df_depenses_onco : 4960 lignes x 13 colonnes
Postes sans type : 0


In [33]:
# Exemple : dépenses 2024 par type (en M€), postes détaillés uniquement, pathologies au niveau localisation
detail = df_depenses_onco[(df_depenses_onco['annee'] == df_depenses_onco['annee'].max())
                          & (df_depenses_onco['type_somme'] == 'Partiel')
                          & (df_depenses_onco['patho_niv3'] == 'Ensemble')]
display((detail.pivot_table(index='type_depense', columns='patho_niv2', values='montant', aggfunc='sum') / 1e6).round(1))

patho_niv2,Autres cancers,Cancer bronchopulmonaire,Cancer colorectal,Cancer de la prostate,Cancer du sein de la femme,Tous cancers
type_depense,,,,,,
HAD (hospitalisation à domicile),526.2,109.0,77.2,40.0,114.3,866.6
Hôpital - actes et consultations externes,342.5,77.5,45.8,68.2,136.3,670.3
Hôpital - liste en sus MCO,4153.3,1269.0,239.9,159.0,1133.0,6954.2
Hôpital - séjours MCO,4714.5,883.8,910.3,569.4,1093.5,8171.5
Prestations - IJ maladie et AT/MP,277.5,53.8,45.0,31.1,261.0,668.3
Prestations - IJ maternité et autres,0.0,0.0,0.0,0.0,0.0,0.0
Prestations - invalidité,298.7,59.1,49.0,13.3,202.0,622.1
Prestations - rentes AT/MP,97.3,53.6,16.6,64.0,7.4,238.9
Psychiatrie,0.0,0.0,0.0,0.0,0.0,0.0


## Base DREES commune aux DF 3 et DF 4 — `df_mh`

**Source** : DREES, « Morbidité hospitalière », tableau 2 : séjours en court séjour (MCO, d'après le PMSI) par tranche de durée et DMS, par pathologie (codes CIM‑10 regroupés), par année et par zone.
Les données sont diffusées sous forme d'archive ZIP jointe au dataset `500_morbidite-hospitaliere`. Elle est téléchargée une seule fois dans `out/raw/`.

**Démarche**
1. Lecture directe du tableau 2 dans le ZIP ; les valeurs `ND` (non diffusées) deviennent `NaN`.
2. `PATHOLOGIE` est séparée en `code_patho` et `libelle_patho`.
3. Géographie harmonisée avec ameli : `niveau` (France / Région / Département), `code_zone` (`FR`, `FRM`, code région ou département) et `libelle_zone`.

`df_mh` contient **toutes les pathologies**, pas seulement les cancers. C'est une table de travail : les DF 3 et DF 4 en sont extraits. On la conserve parce qu'elle contient la ligne `99000 – TOTAL TOUTES CAUSES`, utile pour une jointure (voir DF 4).

In [34]:
mh_url = ('https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/datasets/'
          '500_morbidite-hospitaliere/attachments/morbidite_hospitaliere_tableaux_format_csv_zip')
zip_path = RAW_DIR / 'morbidite_hospitaliere_tableaux_csv.zip'

if not zip_path.exists():
    response = requests.get(mh_url, stream=True, timeout=600)
    if response.status_code == 200:
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"Archive téléchargée : {zip_path} ({zip_path.stat().st_size / 1e6:.1f} Mo)")
    else:
        print(f"Erreur HTTP {response.status_code} : {response.text}")
else:
    print(f"Archive déjà présente : {zip_path}")

with zipfile.ZipFile(zip_path) as z:
    nom_t2 = next(n for n in z.namelist() if 'tableau2' in n.lower())
    with z.open(nom_t2) as f:
        df_mh = pd.read_csv(f, sep=';', encoding='utf-8-sig', na_values=['ND'],
                            dtype={'ANNEE': int, 'Niveau': str, 'PATHOLOGIE': str, 'ZONE': str})

df_mh[['code_patho', 'libelle_patho']] = df_mh['PATHOLOGIE'].str.split('-', n=1, expand=True)

# Géographie harmonisée avec ameli
df_mh['niveau'] = df_mh['Niveau'].map({'France': 'France', 'Régions': 'Région', 'Départements': 'Département'})
df_mh[['code_zone', 'libelle_zone']] = df_mh['ZONE'].str.split(' - ', n=1, expand=True)
df_mh.loc[df_mh['code_zone'] == '3', ['code_zone', 'libelle_zone']] = ['FR', 'France entière']
df_mh.loc[df_mh['code_zone'] == '1', ['code_zone', 'libelle_zone']] = ['FRM', 'France métropolitaine']

df_mh = df_mh.rename(columns={
    'ANNEE': 'annee',
    '<24h hospitalisations programmées': 'sej_moins_24h_programmes',
    '<24h autres hospitalisations': 'sej_moins_24h_autres',
    '<24h ensemble des hospitalisations': 'nb_sejours_moins_24h',
    '1 jour': 'sej_1j', '2 jours': 'sej_2j', '3 jours': 'sej_3j', '4 jours': 'sej_4j', '5 jours': 'sej_5j',
    '6 jours': 'sej_6j', '7 jours': 'sej_7j', '8 jours': 'sej_8j', '9 jours': 'sej_9j',
    '10 à 19 jours': 'sej_10_19j', '20 à 29 jours': 'sej_20_29j', '30 jours et plus': 'sej_30j_plus',
    'Ensemble des hospitalisations': 'nb_sejours',
    'Durée moyenne de séjour (en jours)': 'dms_jours',
}).drop(columns=['Niveau', 'ZONE', 'PATHOLOGIE', 'Données'])

# Pathologies tumorales (chapitre 02) et niveau d'emboîtement des codes
CODES_ONCO = sorted(c for c in df_mh['code_patho'].unique() if c.startswith('02'))
NIVEAU_PATHO = {'02000': 'Chapitre (toutes tumeurs)', '02007': 'Sous-total tumeurs malignes',
                '02025': 'Sous-total tumeurs bénignes'}
df_mh['niveau_patho'] = df_mh['code_patho'].map(NIVEAU_PATHO).fillna('Détail')

print(f"df_mh : {len(df_mh)} lignes, années {df_mh['annee'].min()}-{df_mh['annee'].max()}, "
      f"{len(CODES_ONCO)} codes tumeurs")

Archive déjà présente : out\raw\morbidite_hospitaliere_tableaux_csv.zip
df_mh : 152460 lignes, années 2018-2024, 27 codes tumeurs


## DF 3 — `df_dms_onco` : DMS par pathologie oncologique et par territoire

**Source** : `df_mh` (DREES), codes du chapitre « Tumeurs » `02000` à `02032`.

**Démarche** : simple extraction de `df_mh` (aucune fusion), en gardant la DMS et sa base de calcul.

**Une ligne** = année × zone × code tumeur.

| Colonne | Signification |
|---|---|
| `dms_jours` | DMS DREES, en jours, **calculée sur les séjours de 24 h ou plus** (hors ambulatoire) |
| `nb_sejours_24h_plus` | Nombre de séjours sur lesquels la DMS est calculée (utile pour pondérer une moyenne) |
| `niveau_patho` | `Chapitre (toutes tumeurs)` / `Sous-total tumeurs malignes` / `Sous-total tumeurs bénignes` / `Détail` |

⚠️ **Codes emboîtés** : `02000` = `02007` (malignes) + `02025` (bénignes) + `02031` (in situ) + `02032` (évolution imprévisible). Les localisations `02008` à `02024` sont **incluses** dans `02007`. Pour calculer une DMS agrégée, utiliser une moyenne pondérée par `nb_sejours_24h_plus`, **jamais une moyenne simple**.
⚠️ 2024 est **provisoire**. 2020 est atypique (Covid).
⚠️ Les DMS des DOM et des petits départements reposent sur peu de séjours : elles sont volatiles (jusqu'à 10,9 jours pour le sein à Mayotte en 2024).
⚠️ La DMS est vide (`NaN`) pour 53 lignes, faute de diffusion.

🔗 **Jointures possibles (non réalisées)**
- avec **DF 4** sur (`annee`, `code_zone`, `code_patho`) : correspondance 1 pour 1 (même source) ;
- avec **DF 1** sur (`annee`, `niveau`, `code_zone`) pour le code `02017` (sein) ;
- avec **DF 2** via la **correspondance de pathologies** (fin de notebook).

In [35]:
df_dms_onco = df_mh[df_mh['code_patho'].isin(CODES_ONCO)].copy()
df_dms_onco['nb_sejours_24h_plus'] = df_dms_onco['nb_sejours'] - df_dms_onco['nb_sejours_moins_24h']

df_dms_onco = (df_dms_onco
    [['annee', 'niveau', 'code_zone', 'libelle_zone', 'code_patho', 'libelle_patho', 'niveau_patho',
      'nb_sejours_24h_plus', 'dms_jours']]
    .sort_values(['annee', 'niveau', 'code_zone', 'code_patho'])
    .reset_index(drop=True))

print(f"df_dms_onco : {df_dms_onco.shape[0]} lignes x {df_dms_onco.shape[1]} colonnes")

df_dms_onco : 22869 lignes x 9 colonnes


In [36]:
# Exemple : DMS France entière par localisation maligne, en colonnes par année
display(df_dms_onco[(df_dms_onco['code_zone'] == 'FR') & (df_dms_onco['code_patho'].between('02007', '02024'))]
        .pivot_table(index='libelle_patho', columns='annee', values='dms_jours'))

annee,2018,2019,2020,2021,2022,2023,2024
libelle_patho,,,,,,,
Leucémies,19.68,20.21,20.65,20.21,20.60,20.47,20.22
Mélanome malin de la peau,3.98,3.89,3.84,3.87,4.01,4.14,4.17
T.M. lèvre _bouche_pharynx,9.38,9.25,9.20,8.81,9.16,8.93,9.18
T.M. système nerveux central,10.78,10.79,10.30,10.55,10.52,10.66,10.46
T.M. trachée_bronches_poumon,9.96,9.97,9.62,9.38,9.43,9.43,9.34
T.M.foie et voies biliaires intrahép,7.58,7.57,7.44,7.27,7.63,7.45,7.33
Tumeur maligne de l'estomac,12.81,12.69,12.48,12.23,12.61,12.58,12.26
Tumeur maligne de l'oesophage,12.14,11.95,11.87,11.80,12.46,12.97,12.61
Tumeur maligne de l'utérus,6.09,5.83,5.65,5.35,5.30,5.18,5.09


## DF 4 — `df_charge_onco` : charge hospitalière par pathologie oncologique et par territoire

**Source** : `df_mh` (DREES), codes `02000` à `02032`.

**Démarche** : extraction de `df_mh` (aucune fusion), plus deux indicateurs calculés **à l'intérieur de chaque ligne** :
- `part_ambulatoire` = séjours de moins de 24 h ÷ séjours ;
- `nb_journees_estimees` = séjours de 24 h ou plus × DMS. C'est une approximation du nombre de journées d'hospitalisation, la DREES ne publiant pas les journées.

**Une ligne** = année × zone × code tumeur.

| Colonnes | Signification |
|---|---|
| `nb_sejours` | Total des séjours MCO dont la tumeur est le diagnostic principal |
| `nb_sejours_moins_24h`, `sej_moins_24h_programmes`, `sej_moins_24h_autres` | Séjours de moins de 24 h (ambulatoire), dont programmés |
| `sej_1j` … `sej_30j_plus` | Répartition des séjours de 24 h ou plus par durée |
| `part_ambulatoire`, `nb_journees_estimees` | Indicateurs calculés |

⚠️ **Mêmes emboîtements de codes que le DF 3** : ne pas additionner `02000`, `02007` et les codes détaillés entre eux.
⚠️ Ce sont des **séjours, pas des patients** : un même patient peut avoir plusieurs séjours dans l'année. Les séances de chimiothérapie et de radiothérapie sont comptées dans les séjours de moins de 24 h.
⚠️ Seul le **court séjour (MCO)** est couvert : SSR, HAD et psychiatrie ne sont pas inclus (ils le sont dans les dépenses du DF 2, au niveau national).

🔗 **Jointures possibles (non réalisées)**
- avec `df_mh` filtré sur `code_patho = '99000'` (toutes causes), sur (`annee`, `code_zone`) : **poids de l'oncologie dans l'hôpital** (les tumeurs représentent environ 7,6 % des séjours MCO en 2024) ;
- avec **DF 1** (ou `effectifs`) sur (`annee`, `niveau`, `code_zone`) : séjours pour 1 000 habitants ou par patient ;
- avec **DF 3** : 1 pour 1.

In [37]:
df_charge_onco = df_mh[df_mh['code_patho'].isin(CODES_ONCO)].copy()
df_charge_onco['part_ambulatoire'] = df_charge_onco['nb_sejours_moins_24h'] / df_charge_onco['nb_sejours']
df_charge_onco['nb_journees_estimees'] = ((df_charge_onco['nb_sejours'] - df_charge_onco['nb_sejours_moins_24h'])
                                          * df_charge_onco['dms_jours'])

colonnes_duree = ['sej_1j', 'sej_2j', 'sej_3j', 'sej_4j', 'sej_5j', 'sej_6j', 'sej_7j', 'sej_8j', 'sej_9j',
                  'sej_10_19j', 'sej_20_29j', 'sej_30j_plus']
df_charge_onco = (df_charge_onco
    [['annee', 'niveau', 'code_zone', 'libelle_zone', 'code_patho', 'libelle_patho', 'niveau_patho',
      'nb_sejours', 'nb_sejours_moins_24h', 'sej_moins_24h_programmes', 'sej_moins_24h_autres']
     + colonnes_duree + ['part_ambulatoire', 'nb_journees_estimees']]
    .sort_values(['annee', 'niveau', 'code_zone', 'code_patho'])
    .reset_index(drop=True))

print(f"df_charge_onco : {df_charge_onco.shape[0]} lignes x {df_charge_onco.shape[1]} colonnes")

df_charge_onco : 22869 lignes x 25 colonnes


In [38]:
# Exemple : évolution nationale des séjours pour tumeurs malignes (02007)
display(df_charge_onco[(df_charge_onco['code_zone'] == 'FR') & (df_charge_onco['code_patho'] == '02007')]
        [['annee', 'nb_sejours', 'nb_sejours_moins_24h', 'part_ambulatoire', 'nb_journees_estimees']])

,annee,nb_sejours,nb_sejours_moins_24h,part_ambulatoire,nb_journees_estimees
2728,2018,741196.0,224072.0,0.302311,4374869.04
5995,2019,746302.0,235174.0,0.315119,4288363.92
9262,2020,710449.0,228841.0,0.322108,3963633.84
12529,2021,738743.0,252594.0,0.341924,3860023.06
15796,2022,736601.0,262721.0,0.356667,3795778.80
19063,2023,760039.0,278849.0,0.366888,3839896.20
22330,2024,777931.0,296658.0,0.381342,3811682.16


## 🔗 Récapitulatif des jointures possibles (non réalisées)

### Correspondance des pathologies ameli ↔ DREES
ameli classe les **patients** (cartographie des pathologies, code `top`). La DREES classe les **séjours** par diagnostic principal (codes CIM‑10 regroupés, `code_patho`). Le rapprochement est **approximatif**.

| ameli `patho_niv2` | ameli `top` (préfixe) | DREES `code_patho` |
|---|---|---|
| Cancer du sein de la femme | `CAN_SEI_` | `02017` Tumeur maligne du sein |
| Cancer colorectal | `CAN_CRE_` | `02011` Tumeurs malignes colo‑rectales |
| Cancer bronchopulmonaire | `CAN_BPU_` | `02015` T.M. trachée, bronches, poumon |
| Cancer de la prostate | `CAN_PRO_` | `02019` Tumeur maligne de la prostate |
| Tous cancers | `CAN_CAT_` | `02007` Tumeurs malignes |
| Autres cancers | `CAN_AUT_` | pas de code direct : `02007` − (`02017` + `02011` + `02015` + `02019`) |

Les autres codes DREES (bénignes `02025`‑`02030`, in situ `02031`, etc.) n'ont pas d'équivalent ameli.

### Carte des jointures

| De | Vers | Clés | Précautions | Ce que ça apporte |
|---|---|---|---|---|
| DF 3 | DF 4 | `annee`, `code_zone`, `code_patho` | aucune (1 pour 1) | DMS et volume de séjours dans une même table |
| DF 4 | `df_mh` (`99000`) | `annee`, `code_zone` | — | Part des tumeurs dans l'ensemble des séjours |
| DF 1 | DF 3 / DF 4 | `annee`, `niveau`, `code_zone` + `code_patho = '02017'` | DF 1 : `tsage`, sexe `9`, `Ensemble` | Séjours par patiente, DMS et prévalence |
| DF 2 | DF 3 / DF 4 | `annee` + correspondance ; `code_zone = 'FR'` | DF 2 : `Partiel` ; périmètres différents | Dépense nationale et activité MCO |
| DF 1, 3, 4 | `referentiel-departements` (ameli) | code département | — | Libellés de départements |

### Points de vigilance transversaux
- **Périodes** : ameli 2015‑2024, DREES 2018‑2024 (2024 provisoire). Une jointure ameli × DREES ne garde que 2018‑2024.
- **Périmètres** : DREES = **séjours** MCO avec un cancer en diagnostic principal. ameli = **patients** actifs ou sous surveillance, et **tous** les soins remboursés affectés au cancer. On peut les comparer côte à côte, mais **pas diviser l'un par l'autre** (pas de « coût par séjour »).
- **Géographie** : ameli et DREES sont rattachés en principe au domicile du patient (d'après la documentation des sources). Pas de niveau ville : le secret statistique empêche toute diffusion par pathologie à l'échelle communale.
- **Années atypiques** : 2020 (Covid, environ −10 % de séjours pour tumeurs).
- **DOM** : faibles effectifs, prévalence très basse, DMS volatiles. Les analyser à part.

# Exploration des DataFrames

## Exploration — DF 1 : `df_patients_sein`

Patients atteints d'un cancer du sein (ameli `effectifs`), par année, sous‑pathologie (ensemble / actif / sous surveillance), classe d'âge, sexe et territoire (France, région, département), 2015‑2024.

### Signification de colonnes clés

#### `top` : code technique de la pathologie
Dictionnaire ameli : *« libellé technique de la pathologie (ou traitement chronique ou épisode de soins) »*.
C'est l'identifiant de la pathologie dans la **cartographie des pathologies** de l'Assurance Maladie, sous la forme `GROUPE_SOUSGROUPE_DETAIL` :

| `top` | `patho_niv3` | Signification |
|---|---|---|
| `CAN_SEI_CAT` | `Ensemble` | **Toutes** les patientes ayant un cancer du sein, quel que soit le stade (CAT = catégorie complète, niveau 2 de l'arborescence) |
| `CAN_SEI_ACT` | `Cancer du sein de la femme actif` | Cancer en **phase active** de prise en charge (traitement ou hospitalisation récents) |
| `CAN_SEI_SUR` | `Cancer du sein de la femme sous surveillance` | Cancer plus ancien, en **phase de suivi**, sans prise en charge active récente |

- `CAN` = groupe *Cancers* (`patho_niv1`), `SEI` = *sein* (`patho_niv2`), le suffixe = niveau de détail (`patho_niv3`).
- ✅ Vérifié : **actif + sous surveillance = ensemble** (France 2024 : 264 290 + 464 690 = 728 980 patientes). Il ne faut donc jamais additionner `CAN_SEI_CAT` avec les deux autres.
- Les autres cancers suivent le même schéma : `CAN_CRE` (colorectal), `CAN_BPU` (bronchopulmonaire), `CAN_PRO` (prostate), `CAN_AUT` (autres cancers), `CAN_CAT_CAT` (tous cancers).
- 🔗 `top` est la **clé commune** entre `effectifs`, `depenses` (DF 2) et le référentiel ameli `referentiel-pathologies`.
- ℹ️ Les critères exacts (ALD, hospitalisations, traitements, période observée) qui classent une patiente en « actif » ou « sous surveillance » sont décrits dans la méthodologie de la cartographie Cnam, fichier « Définitions des pathologies ». Ils ne sont pas repris ici.

#### `prevalence_pct` (colonne `prev` d'origine) : prévalence
Définition ameli : *« proportion de patients pris en charge, à un moment donné, pour une pathologie dans une population (en %) »*.

**prevalence_pct = nb_patients ÷ population × 100**. Vérifié : France 2024, 728 980 ÷ 35 171 580 × 100 = **2,073 %**.

- **`population` (colonne `npop` d'origine)** : population de la cartographie, c'est‑à‑dire les bénéficiaires de l'assurance maladie obligatoire, tous régimes, ayant eu au moins un soin remboursé ou un séjour à l'hôpital dans l'année (67,6 millions en 2024). Ce **n'est pas** la population INSEE.
- ⚠️ **Pour le cancer du sein, `population` ne compte que les femmes** (35,2 millions en 2024), y compris sur les lignes `sexe = 9`. Les lignes `sexe = 9` (tous sexes) sont **identiques** aux lignes `sexe = 2` (femmes) : la table contient donc chaque information en double. Il faut filtrer sur **un seul** des deux codes.
- ⚠️ **Prévalence ≠ incidence** : on compte toutes les patientes prises en charge dans l'année (cancers actifs + sous surveillance), pas les nouveaux cas.
- ⚠️ **Prévalence brute** : elle dépend de la structure par âge du territoire. Pour comparer des territoires, raisonner par classe d'âge (`cla_age_5`) ou standardiser.
- ⚠️ **Secret statistique** : en dessous de 11 patientes, `nb_patients` et `prevalence_pct` sont vides (`NaN`, « NS » dans la source) ; ce n'est pas zéro. Les effectifs diffusés sont arrondis à la dizaine.

### Notes d'exploration

#### 1. Premier diagnostic de `df_patients_sein` (`head`, `isna`, `duplicated`, `info`, valeurs uniques)
- **151 200 lignes × 15 colonnes**, **aucun doublon** (`duplicated()` = 0).
- **Types** : `annee` en entier, `population` en entier, `nb_patients` et `prevalence_pct` en décimal, le reste en texte.
- **Valeurs manquantes** : seulement `nb_patients` et `prevalence_pct`, sur **47 536 lignes (31 %)**. Ce sont les petits effectifs masqués par le secret statistique (moins de 11 patientes), surtout dans les jeunes tranches d'âge et les petits territoires. Ce ne sont pas des zéros.
- **Valeurs uniques** :

| Colonne | Valeurs uniques | Constat |
|---|---:|---|
| `annee` | 10 | 2015 à 2024, aucune année manquante |
| `niveau` | 3 | France, Région, Département |
| `code_region` | 19 | 18 régions + `99` (France) |
| `dept` | 102 | 101 départements + `999` (pas de département) |
| `patho_niv2` | 1 | toujours « Cancer du sein de la femme » → **inutile** |
| `patho_niv3` | 3 | Ensemble, actif, sous surveillance |
| `top` | 3 | `CAN_SEI_CAT/ACT/SUR`, même information que `patho_niv3` → **redondant** |
| `cla_age_5` / `libelle_classe_age` | 21 | même information, code ou libellé → **redondant** |
| `sexe` / `libelle_sexe` | 2 | `2` = femmes, `9` = tous sexes → **redondant** |

#### 2. Nettoyage
- Colonnes supprimées : `patho_niv2` (constante), `top`, `cla_age_5` et `sexe` (doublons de `patho_niv3`, `libelle_classe_age` et `libelle_sexe`).
- ⚠️ `top` est la clé commune avec le DF de dépenses (DF 2). Pour une future jointure, on la retrouve à partir de `patho_niv3` (Ensemble → `CAN_SEI_CAT`, actif → `CAN_SEI_ACT`, sous surveillance → `CAN_SEI_SUR`).

#### 3. Un DataFrame par niveau de granularité
- Essai d'un index hiérarchique niveau > région > département, abandonné : peu pratique pour l'analyse.
- **Choix retenu** : découper `df_patients_sein` en 3 DF selon `niveau`, en gardant les **totaux publiés par ameli**. Un recalcul par somme des départements perdrait les effectifs masqués (2024, femmes de 25 à 29 ans : 920 au total France publié contre 650 en additionnant les départements).
- Codes remplacés par des **noms** grâce aux référentiels ameli (`referentiel-regions`, `referentiel-departements`). Aucun code sans nom.

| DataFrame | Lignes | Colonnes géographiques | Lignes sans `nb_patients` |
|---|---:|---|---:|
| `df_sein_france` | 1 260 | `territoire` | 158 (13 %) |
| `df_sein_region` | 22 680 | `nom_region` | 6 706 (30 %) |
| `df_sein_dept` | 127 260 | `nom_region`, `dept`, `nom_dept` | 40 672 (32 %) |

Contrôle : 1 260 + 22 680 + 127 260 = 151 200 lignes, et 158 + 6 706 + 40 672 = 47 536 valeurs manquantes. Rien n'est perdu.

#### 4. Points d'attention pour la suite
- **« tous sexes » = « femmes »** : `duplicated()` renvoie 0 car `libelle_sexe` diffère, mais les chiffres sont identiques. Filtrer sur une seule valeur de `libelle_sexe` avant tout calcul.
- **Totaux emboîtés** : `Ensemble` = actif + sous surveillance, et `tous âges` = somme des tranches d'âge. Ne pas les additionner entre eux.
- **DOM** : présents à la fois dans `df_sein_region` et dans `df_sein_dept`, avec les mêmes chiffres.
- **Valeurs manquantes** : à traiter explicitement selon l'analyse (exclure, ou travailler sur `tous âges` où elles sont rares). Ne jamais les remplacer par 0.

In [41]:
display(df_patients_sein.head(10))

,annee,niveau,code_zone,code_region,dept,patho_niv2,patho_niv3,top,cla_age_5,libelle_classe_age,sexe,libelle_sexe,nb_patients,population,prevalence_pct
0,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,00-04,de 0 à 4 ans,2,femmes,NaN,17430,NaN
1,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,05-09,de 5 à 9 ans,2,femmes,NaN,18840,NaN
2,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,10-14,de 10 à 14 ans,2,femmes,NaN,18500,NaN
3,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,15-19,de 15 à 19 ans,2,femmes,NaN,17730,NaN
4,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,20-24,de 20 à 24 ans,2,femmes,NaN,15400,NaN
5,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,25-29,de 25 à 29 ans,2,femmes,NaN,16930,NaN
6,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,30-34,de 30 à 34 ans,2,femmes,20.0,18580,0.108
7,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,35-39,de 35 à 39 ans,2,femmes,40.0,19060,0.205
8,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,40-44,de 40 à 44 ans,2,femmes,100.0,20900,0.455
9,2015,Département,01,84,01,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,45-49,de 45 à 49 ans,2,femmes,180.0,20720,0.850


In [42]:
df_patients_sein.isna().sum()

annee                     0
niveau                    0
code_zone                 0
code_region               0
dept                      0
patho_niv2                0
patho_niv3                0
top                       0
cla_age_5                 0
libelle_classe_age        0
sexe                      0
libelle_sexe              0
nb_patients           47536
population                0
prevalence_pct        47536
dtype: int64

In [43]:
df_patients_sein.duplicated().sum()

np.int64(0)

In [44]:
df_patients_sein.info()

<class 'pandas.DataFrame'>
RangeIndex: 151200 entries, 0 to 151199
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   annee               151200 non-null  int32  
 1   niveau              151200 non-null  str    
 2   code_zone           151200 non-null  str    
 3   code_region         151200 non-null  str    
 4   dept                151200 non-null  str    
 5   patho_niv2          151200 non-null  str    
 6   patho_niv3          151200 non-null  str    
 7   top                 151200 non-null  str    
 8   cla_age_5           151200 non-null  str    
 9   libelle_classe_age  151200 non-null  str    
 10  sexe                151200 non-null  str    
 11  libelle_sexe        151200 non-null  str    
 12  nb_patients         103664 non-null  float64
 13  population          151200 non-null  int64  
 14  prevalence_pct      103664 non-null  float64
dtypes: float64(2), int32(1), int64(1), str(11)
me

In [45]:
# Valeurs uniques de chaque colonne (au-delà de 50 valeurs, seules les 50 premières sont affichées)
for col in df_patients_sein.columns:
    valeurs = df_patients_sein[col].unique()
    print (f"{col} : {len(valeurs)} valeurs uniques")
    if len(valeurs) <= 50:
        print(f'les valeurs uniques sont : {valeurs.tolist()}')

annee : 10 valeurs uniques
les valeurs uniques sont : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
niveau : 3 valeurs uniques
les valeurs uniques sont : ['Département', 'France', 'Région']
code_zone : 102 valeurs uniques
code_region : 19 valeurs uniques
les valeurs uniques sont : ['84', '32', '93', '44', '76', '28', '75', '24', '27', '53', '94', '52', '11', '01', '02', '03', '04', '06', '99']
dept : 102 valeurs uniques
patho_niv2 : 1 valeurs uniques
les valeurs uniques sont : ['Cancer du sein de la femme']
patho_niv3 : 3 valeurs uniques
les valeurs uniques sont : ['Cancer du sein de la femme actif', 'Ensemble', 'Cancer du sein de la femme sous surveillance']
top : 3 valeurs uniques
les valeurs uniques sont : ['CAN_SEI_ACT', 'CAN_SEI_CAT', 'CAN_SEI_SUR']
cla_age_5 : 21 valeurs uniques
les valeurs uniques sont : ['00-04', '05-09', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85-89',

In [46]:
df_patients_sein.drop(columns=['patho_niv2', 'top','cla_age_5', 'sexe' ], inplace=True)

In [47]:
df_patients_sein.head(10)

,annee,niveau,code_zone,code_region,dept,patho_niv3,libelle_classe_age,libelle_sexe,nb_patients,population,prevalence_pct
0,2015,Département,01,84,01,Cancer du sein de la femme actif,de 0 à 4 ans,femmes,NaN,17430,NaN
1,2015,Département,01,84,01,Cancer du sein de la femme actif,de 5 à 9 ans,femmes,NaN,18840,NaN
2,2015,Département,01,84,01,Cancer du sein de la femme actif,de 10 à 14 ans,femmes,NaN,18500,NaN
3,2015,Département,01,84,01,Cancer du sein de la femme actif,de 15 à 19 ans,femmes,NaN,17730,NaN
4,2015,Département,01,84,01,Cancer du sein de la femme actif,de 20 à 24 ans,femmes,NaN,15400,NaN
5,2015,Département,01,84,01,Cancer du sein de la femme actif,de 25 à 29 ans,femmes,NaN,16930,NaN
6,2015,Département,01,84,01,Cancer du sein de la femme actif,de 30 à 34 ans,femmes,20.0,18580,0.108
7,2015,Département,01,84,01,Cancer du sein de la femme actif,de 35 à 39 ans,femmes,40.0,19060,0.205
8,2015,Département,01,84,01,Cancer du sein de la femme actif,de 40 à 44 ans,femmes,100.0,20900,0.455
9,2015,Département,01,84,01,Cancer du sein de la femme actif,de 45 à 49 ans,femmes,180.0,20720,0.850


### DF 1 découpé par niveau de granularité

Pour faciliter les analyses, `df_patients_sein` est découpé en **3 DataFrames**, un par échelle géographique. On reprend les **totaux publiés par ameli** (lignes France et Région de la source) plutôt que de les recalculer en additionnant les départements. Une somme de départements perdrait les petits effectifs masqués par le secret statistique : en 2024, femmes de 25 à 29 ans, 920 patientes au total France publié contre 650 en additionnant les départements.

| DataFrame | Contenu | Colonnes géographiques gardées | Lignes |
|---|---|---|---:|
| `df_sein_france` | France entière | `territoire` (= « France ») | 1 260 |
| `df_sein_region` | 18 régions (dont 5 DOM) | `nom_region` | 22 680 |
| `df_sein_dept` | 101 départements | `nom_region`, `dept` (numéro), `nom_dept` | 127 260 |

Les noms des régions et départements viennent des référentiels ameli `referentiel-regions` et `referentiel-departements`, qui utilisent les mêmes codes que les données. Les codes de région sont remplacés par leur nom ; pour les départements, on garde le numéro et le nom.

🔗 Pour joindre avec les autres DF (qui utilisent `code_zone`), passer par le numéro `dept` pour les départements. Pour les régions, retrouver le code via `referentiel-regions`.

Les autres colonnes sont identiques dans les 3 DF : `annee`, `patho_niv3`, `libelle_classe_age`, `libelle_sexe`, `nb_patients`, `population`, `prevalence_pct`.

⚠️ Les lignes « tous sexes » sont conservées. Pour le cancer du sein, elles sont **identiques** aux lignes « femmes » : filtrer sur une seule valeur de `libelle_sexe` avant tout calcul.
ℹ️ Dans les DOM, la région et le département se confondent : on retrouve les mêmes chiffres dans `df_sein_region` (codes `01` à `06`) et dans `df_sein_dept` (codes `971` à `976`).

In [48]:
# Un DataFrame par niveau de granularité, à partir des totaux publiés par ameli (pas de recalcul par somme)
base = df_patients_sein.reset_index() if 'niveau' in df_patients_sein.index.names else df_patients_sein.copy()

colonnes_geo = ['niveau', 'code_zone', 'code_region', 'dept']
colonnes_donnees = [c for c in base.columns if c not in colonnes_geo]

def extraire(niveau, colonnes_geo_a_garder):
    df = base[base['niveau'] == niveau][colonnes_geo_a_garder + colonnes_donnees]
    if colonnes_geo_a_garder:
        df = df.sort_values(colonnes_geo_a_garder, kind='stable')
    return df.reset_index(drop=True)

df_sein_france = extraire('France', [])
df_sein_region = extraire('Région', ['code_region'])
df_sein_dept = extraire('Département', ['code_region', 'dept'])

for nom, df in [('df_sein_france', df_sein_france), ('df_sein_region', df_sein_region), ('df_sein_dept', df_sein_dept)]:
    print(f"{nom} : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Total : {len(df_sein_france) + len(df_sein_region) + len(df_sein_dept)} lignes (df_patients_sein : {len(base)})")

display(df_sein_region.head(5))

df_sein_france : 1260 lignes x 7 colonnes
df_sein_region : 22680 lignes x 8 colonnes
df_sein_dept : 127260 lignes x 9 colonnes
Total : 151200 lignes (df_patients_sein : 151200)


,code_region,annee,patho_niv3,libelle_classe_age,libelle_sexe,nb_patients,population,prevalence_pct
0,01,2015,Cancer du sein de la femme actif,de 0 à 4 ans,femmes,NaN,10580,NaN
1,01,2015,Cancer du sein de la femme actif,de 5 à 9 ans,femmes,NaN,12320,NaN
2,01,2015,Cancer du sein de la femme actif,de 10 à 14 ans,femmes,NaN,13070,NaN
3,01,2015,Cancer du sein de la femme actif,de 15 à 19 ans,femmes,NaN,13290,NaN
4,01,2015,Cancer du sein de la femme actif,de 20 à 24 ans,femmes,NaN,10280,NaN


In [49]:
# Noms des territoires : référentiels ameli des régions et des départements (mêmes codes que les données)
ref_regions = export_ameli('referentiel-regions', 'reg_code IS NOT NULL')
ref_depts = export_ameli('referentiel-departements', 'dep_code IS NOT NULL')

nom_region = dict(zip(ref_regions['reg_code'], ref_regions['reg_name']))   # ex. '11' -> 'Île-de-France'
nom_dept = dict(zip(ref_depts['dep_code'], ref_depts['dep_name']))         # ex. '75' -> 'Paris'

# France : seulement le nom
if 'territoire' not in df_sein_france.columns:
    df_sein_france.insert(0, 'territoire', 'France')

# Régions : le code de région est remplacé par son nom (pop retire la colonne et renvoie ses valeurs)
for df in (df_sein_region, df_sein_dept):
    if 'code_region' in df.columns:
        df.drop(columns=['nom_region'], errors='ignore', inplace=True)
        df.insert(0, 'nom_region', df.pop('code_region').map(nom_region))

# Départements : on garde le numéro et on ajoute le nom juste après
df_sein_dept.drop(columns=['nom_dept'], errors='ignore', inplace=True)
df_sein_dept.insert(df_sein_dept.columns.get_loc('dept') + 1, 'nom_dept', df_sein_dept['dept'].map(nom_dept))

# Contrôle : aucun territoire sans nom
print("Régions sans nom :", df_sein_region['nom_region'].isna().sum())
print("Départements sans nom :", df_sein_dept[['nom_region', 'nom_dept']].isna().sum().sum())

display(df_sein_dept.head(10))
display(df_sein_region.head(10))
display(df_sein_france.head(10))

referentiel-regions : 27 enregistrements récupérés
referentiel-departements : 110 enregistrements récupérés
Régions sans nom : 0
Départements sans nom : 0


,nom_region,dept,nom_dept,annee,patho_niv3,libelle_classe_age,libelle_sexe,nb_patients,population,prevalence_pct
0,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 0 à 4 ans,femmes,NaN,10580,NaN
1,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 5 à 9 ans,femmes,NaN,12320,NaN
2,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 10 à 14 ans,femmes,NaN,13070,NaN
3,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 15 à 19 ans,femmes,NaN,13290,NaN
4,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 20 à 24 ans,femmes,NaN,10280,NaN
5,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 25 à 29 ans,femmes,NaN,10150,NaN
6,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 30 à 34 ans,femmes,10.0,11640,0.120
7,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 35 à 39 ans,femmes,30.0,12290,0.252
8,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 40 à 44 ans,femmes,80.0,16490,0.503
9,Guadeloupe,971,Guadeloupe,2015,Cancer du sein de la femme actif,de 45 à 49 ans,femmes,120.0,17010,0.676


,nom_region,annee,patho_niv3,libelle_classe_age,libelle_sexe,nb_patients,population,prevalence_pct
0,Guadeloupe,2015,Cancer du sein de la femme actif,de 0 à 4 ans,femmes,NaN,10580,NaN
1,Guadeloupe,2015,Cancer du sein de la femme actif,de 5 à 9 ans,femmes,NaN,12320,NaN
2,Guadeloupe,2015,Cancer du sein de la femme actif,de 10 à 14 ans,femmes,NaN,13070,NaN
3,Guadeloupe,2015,Cancer du sein de la femme actif,de 15 à 19 ans,femmes,NaN,13290,NaN
4,Guadeloupe,2015,Cancer du sein de la femme actif,de 20 à 24 ans,femmes,NaN,10280,NaN
5,Guadeloupe,2015,Cancer du sein de la femme actif,de 25 à 29 ans,femmes,NaN,10150,NaN
6,Guadeloupe,2015,Cancer du sein de la femme actif,de 30 à 34 ans,femmes,10.0,11640,0.120
7,Guadeloupe,2015,Cancer du sein de la femme actif,de 35 à 39 ans,femmes,30.0,12290,0.252
8,Guadeloupe,2015,Cancer du sein de la femme actif,de 40 à 44 ans,femmes,80.0,16490,0.503
9,Guadeloupe,2015,Cancer du sein de la femme actif,de 45 à 49 ans,femmes,120.0,17010,0.676


,territoire,annee,patho_niv3,libelle_classe_age,libelle_sexe,nb_patients,population,prevalence_pct
0,France,2015,Cancer du sein de la femme actif,de 0 à 4 ans,femmes,NaN,1887420,NaN
1,France,2015,Cancer du sein de la femme actif,de 5 à 9 ans,femmes,NaN,1934750,NaN
2,France,2015,Cancer du sein de la femme actif,de 10 à 14 ans,femmes,NaN,1883250,NaN
3,France,2015,Cancer du sein de la femme actif,de 15 à 19 ans,femmes,30.0,1859550,0.002
4,France,2015,Cancer du sein de la femme actif,de 20 à 24 ans,femmes,110.0,1865930,0.006
5,France,2015,Cancer du sein de la femme actif,de 25 à 29 ans,femmes,610.0,2052590,0.030
6,France,2015,Cancer du sein de la femme actif,de 30 à 34 ans,femmes,2230.0,2113370,0.105
7,France,2015,Cancer du sein de la femme actif,de 35 à 39 ans,femmes,5110.0,2058890,0.248
8,France,2015,Cancer du sein de la femme actif,de 40 à 44 ans,femmes,11700.0,2200070,0.532
9,France,2015,Cancer du sein de la femme actif,de 45 à 49 ans,femmes,19080.0,2214300,0.862


In [50]:
df_sein_france.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   territoire          1260 non-null   str    
 1   annee               1260 non-null   int32  
 2   patho_niv3          1260 non-null   str    
 3   libelle_classe_age  1260 non-null   str    
 4   libelle_sexe        1260 non-null   str    
 5   nb_patients         1102 non-null   float64
 6   population          1260 non-null   int64  
 7   prevalence_pct      1102 non-null   float64
dtypes: float64(2), int32(1), int64(1), str(4)
memory usage: 144.1 KB


In [51]:
df_sein_region.info()

<class 'pandas.DataFrame'>
RangeIndex: 22680 entries, 0 to 22679
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nom_region          22680 non-null  str    
 1   annee               22680 non-null  int32  
 2   patho_niv3          22680 non-null  str    
 3   libelle_classe_age  22680 non-null  str    
 4   libelle_sexe        22680 non-null  str    
 5   nb_patients         15974 non-null  float64
 6   population          22680 non-null  int64  
 7   prevalence_pct      15974 non-null  float64
dtypes: float64(2), int32(1), int64(1), str(4)
memory usage: 2.7 MB


In [52]:
df_sein_dept.info()

<class 'pandas.DataFrame'>
RangeIndex: 127260 entries, 0 to 127259
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   nom_region          127260 non-null  str    
 1   dept                127260 non-null  str    
 2   nom_dept            127260 non-null  str    
 3   annee               127260 non-null  int32  
 4   patho_niv3          127260 non-null  str    
 5   libelle_classe_age  127260 non-null  str    
 6   libelle_sexe        127260 non-null  str    
 7   nb_patients         86588 non-null   float64
 8   population          127260 non-null  int64  
 9   prevalence_pct      86588 non-null   float64
dtypes: float64(2), int32(1), int64(1), str(6)
memory usage: 18.7 MB


## Exploration — DF 2 : `df_depenses_onco`

Dépenses remboursées par l'Assurance Maladie (ameli `depenses`) pour chaque pathologie oncologique et chaque poste de dépense (hôpital, ville, prestations en espèces), niveau France, 2015‑2024.

### Notes d'exploration

#### 1. Premier diagnostic de `df_depenses_onco`
- **4 960 lignes × 13 colonnes**, **aucun doublon**, **aucune valeur manquante**.
- **Types** : `annee`, `montant`, `nb_patients_pathologie` et `n_recourant_au_poste` en entier ; `montant_moy` en décimal ; le reste en texte.

| Colonne | Valeurs uniques | Constat |
|---|---:|---|
| `annee` | 10 | 2015 à 2024, aucune année manquante |
| `patho_niv2` | 6 | 5 localisations + « Tous cancers » |
| `patho_niv3` | 11 | `Ensemble` + actif / sous surveillance pour chacune des 5 localisations (« Tous cancers » n'existe qu'en `Ensemble`) |
| `top` | 16 | même information que (`patho_niv2`, `patho_niv3`) → **redondant** (vérifié) |
| `dep_niv_1` | 4 | Hospitalisations, Soins de ville, Prestations en espèces, et `Dépenses` (= total général) |
| `type_depense` | 26 | 22 postes détaillés + 4 totaux (`TOTAL …`) |
| `secteur` | 3 | Public, Privé, Non précisé |
| `dep_niv_2` | 31 | même information que (`type_depense`, `secteur`) → **redondant** (vérifié) |
| `type_somme` | 2 | `Partiel` (poste détaillé) / `Total` (sous-total ou total) |
| `nb_patients_pathologie` | 160 | une valeur par année et par pathologie (10 × 16) |

- **Postes toujours à 0 €** pour les cancers : `Psychiatrie`, `Ville - sages-femmes`, `Prestations - IJ maternité et autres`.
- **`montant_moy` = dépense moyenne par patient de la pathologie** : `montant ÷ nb_patients_pathologie` (vérifié sur 4 958 lignes sur 4 960, à 1 € près). Ce n'est **pas** la dépense par patient ayant eu recours au poste (`n_recourant_au_poste`).

#### 2. Nettoyage
- Colonnes supprimées : `top` et `dep_niv_2`, après vérification qu'elles sont redondantes (fonction `correspondance_1_1`, réutilisée pour les DF suivants).

#### 3. Granularité
- Pas de découpage géographique : ce jeu de données n'existe **qu'au niveau France**.

#### 4. Points d'attention pour la suite
- **Double comptage** : garder `type_somme == 'Partiel'` pour additionner des postes. Les lignes `Total` et `dep_niv_1 == 'Dépenses'` sont déjà des sommes.
- **Totaux emboîtés** : `Ensemble` = actif + sous surveillance, et « Tous cancers » = somme des 5 localisations.
- Ce sont des montants **remboursés** et **affectés** à la pathologie par la méthode de la cartographie : ni un coût complet, ni le reste à charge.

In [53]:
display(df_depenses_onco.head(10))

,annee,patho_niv2,patho_niv3,top,dep_niv_1,type_depense,secteur,dep_niv_2,type_somme,montant,nb_patients_pathologie,n_recourant_au_poste,montant_moy
0,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Dépenses,TOTAL général,Non précisé,Total des dépenses remboursées,Total,7696571777,818710,818710,9401.0
1,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),Hôpital - actes et consultations externes,Public,Actes et consultations externes MCO secteur pu...,Partiel,158869055,818710,587500,194.0
2,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),HAD (hospitalisation à domicile),Privé,Hospitalisations en HAD secteur privé remboursées,Partiel,85262673,818710,10340,104.0
3,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),HAD (hospitalisation à domicile),Public,Hospitalisations en HAD secteur public rembour...,Partiel,131476088,818710,13670,161.0
4,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),SSR (soins de suite et de réadaptation),Privé,Hospitalisations en SSR secteur privé remboursées,Partiel,84313318,818710,27140,103.0
5,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),SSR (soins de suite et de réadaptation),Public,Hospitalisations en SSR secteur public rembour...,Partiel,237697077,818710,48300,290.0
6,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),Psychiatrie,Privé,Hospitalisations en psychiatrie secteur privé ...,Partiel,0,818710,1590,0.0
7,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),Psychiatrie,Public,Hospitalisations en psychiatrie secteur public...,Partiel,0,818710,3800,0.0
8,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),Hôpital - liste en sus MCO,Privé,Hospitalisations liste en sus MCO secteur priv...,Partiel,117607025,818710,31830,144.0
9,2015,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,Hospitalisations (tous secteurs),Hôpital - liste en sus MCO,Public,Hospitalisations liste en sus MCO secteur publ...,Partiel,633260473,818710,96190,773.0


In [54]:
df_depenses_onco.isna().sum()

annee                     0
patho_niv2                0
patho_niv3                0
top                       0
dep_niv_1                 0
type_depense              0
secteur                   0
dep_niv_2                 0
type_somme                0
montant                   0
nb_patients_pathologie    0
n_recourant_au_poste      0
montant_moy               0
dtype: int64

In [55]:
df_depenses_onco.duplicated().sum()

np.int64(0)

In [56]:
df_depenses_onco.info()

<class 'pandas.DataFrame'>
RangeIndex: 4960 entries, 0 to 4959
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   annee                   4960 non-null   int32  
 1   patho_niv2              4960 non-null   str    
 2   patho_niv3              4960 non-null   str    
 3   top                     4960 non-null   str    
 4   dep_niv_1               4960 non-null   str    
 5   type_depense            4960 non-null   str    
 6   secteur                 4960 non-null   str    
 7   dep_niv_2               4960 non-null   str    
 8   type_somme              4960 non-null   str    
 9   montant                 4960 non-null   int64  
 10  nb_patients_pathologie  4960 non-null   int64  
 11  n_recourant_au_poste    4960 non-null   int64  
 12  montant_moy             4960 non-null   float64
dtypes: float64(1), int32(1), int64(3), str(8)
memory usage: 1.2 MB


In [57]:
# Valeurs uniques de chaque colonne (la liste n'est affichée que s'il y a 50 valeurs ou moins)
for col in df_depenses_onco.columns:
    valeurs = df_depenses_onco[col].unique()
    print (f"{col} : {len(valeurs)} valeurs uniques")
    if len(valeurs) <= 50:
        print(f'les valeurs uniques sont : {valeurs.tolist()}')

annee : 10 valeurs uniques
les valeurs uniques sont : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
patho_niv2 : 6 valeurs uniques
les valeurs uniques sont : ['Autres cancers', 'Cancer bronchopulmonaire', 'Tous cancers', 'Cancer colorectal', 'Cancer de la prostate', 'Cancer du sein de la femme']
patho_niv3 : 11 valeurs uniques
les valeurs uniques sont : ['Autres cancers actifs', 'Ensemble', 'Autres cancers sous surveillance', 'Cancer bronchopulmonaire actif', 'Cancer bronchopulmonaire sous surveillance', 'Cancer colorectal actif', 'Cancer colorectal sous surveillance', 'Cancer de la prostate actif', 'Cancer de la prostate sous surveillance', 'Cancer du sein de la femme actif', 'Cancer du sein de la femme sous surveillance']
top : 16 valeurs uniques
les valeurs uniques sont : ['CAN_AUT_ACT', 'CAN_AUT_CAT', 'CAN_AUT_SUR', 'CAN_BPU_ACT', 'CAN_BPU_CAT', 'CAN_BPU_SUR', 'CAN_CAT_CAT', 'CAN_CRE_ACT', 'CAN_CRE_CAT', 'CAN_CRE_SUR', 'CAN_PRO_ACT', 'CAN_PRO_CAT', 'CAN_PRO_SUR', 'CA

In [58]:
# Outil réutilisé pour les DF suivants : deux colonnes (ou groupes de colonnes) portent-elles exactement la même information ?
def correspondance_1_1(df, col_a, cols_b):
    cols_b = [cols_b] if isinstance(cols_b, str) else cols_b
    return bool(df.groupby(col_a)[cols_b].nunique().max().max() == 1
                and df.groupby(cols_b)[col_a].nunique().max() == 1)

print("top <-> (patho_niv2, patho_niv3) :", correspondance_1_1(df_depenses_onco, 'top', ['patho_niv2', 'patho_niv3']))
print("dep_niv_2 <-> (type_depense, secteur) :", correspondance_1_1(df_depenses_onco, 'dep_niv_2', ['type_depense', 'secteur']))

top <-> (patho_niv2, patho_niv3) : True
dep_niv_2 <-> (type_depense, secteur) : True


In [59]:
# Postes de dépense toujours à 0 € pour les cancers (toutes années, toutes pathologies)
total_par_type = df_depenses_onco.groupby('type_depense')['montant'].sum()
print(total_par_type[total_par_type == 0].index.tolist())

['Prestations - IJ maternité et autres', 'Psychiatrie', 'Ville - sages-femmes']


In [60]:
df_depenses_onco.drop(columns=['top', 'dep_niv_2'], inplace=True)

In [61]:
df_depenses_onco.head(10)

,annee,patho_niv2,patho_niv3,dep_niv_1,type_depense,secteur,type_somme,montant,nb_patients_pathologie,n_recourant_au_poste,montant_moy
0,2015,Autres cancers,Autres cancers actifs,Dépenses,TOTAL général,Non précisé,Total,7696571777,818710,818710,9401.0
1,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),Hôpital - actes et consultations externes,Public,Partiel,158869055,818710,587500,194.0
2,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),HAD (hospitalisation à domicile),Privé,Partiel,85262673,818710,10340,104.0
3,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),HAD (hospitalisation à domicile),Public,Partiel,131476088,818710,13670,161.0
4,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),SSR (soins de suite et de réadaptation),Privé,Partiel,84313318,818710,27140,103.0
5,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),SSR (soins de suite et de réadaptation),Public,Partiel,237697077,818710,48300,290.0
6,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),Psychiatrie,Privé,Partiel,0,818710,1590,0.0
7,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),Psychiatrie,Public,Partiel,0,818710,3800,0.0
8,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),Hôpital - liste en sus MCO,Privé,Partiel,117607025,818710,31830,144.0
9,2015,Autres cancers,Autres cancers actifs,Hospitalisations (tous secteurs),Hôpital - liste en sus MCO,Public,Partiel,633260473,818710,96190,773.0


## Exploration — DF 3 : `df_dms_onco`

Durée moyenne de séjour (DREES, séjours de 24 h ou plus) par code tumeur (02000 à 02032) et par territoire (France, région, département), 2018‑2024.

### Notes d'exploration

#### 1. Premier diagnostic de `df_dms_onco`
- **22 869 lignes × 9 colonnes**, **aucun doublon**.
- **Valeurs manquantes** : `dms_jours` sur **53 lignes**, `nb_sejours_24h_plus` sur 4. Elles sont toutes au niveau Région ou Département, **aucune au niveau France**. Il s'agit surtout de « Tumeurs bénignes de la peau » (36 lignes), puis de quelques cas isolés : tumeur bénigne du sein (9), larynx (3), pancréas (2), mélanome (2), estomac (1).

| Colonne | Valeurs uniques | Constat |
|---|---:|---|
| `annee` | 7 | 2018 à 2024 (2024 provisoire) |
| `niveau` | 3 | France, Région, Département |
| `code_zone` | 103 | FR, FRM, 18 régions, 101 départements ; certains codes de région et de département sont identiques (`11`, `01`…), d'où moins de 121 valeurs |
| `libelle_zone` | 117 | dans les DOM, la région et le département ont le même nom |
| `code_patho` / `libelle_patho` | 27 | même information, code ou libellé → **redondant** (vérifié) |
| `niveau_patho` | 4 | Chapitre, Sous-total malignes, Sous-total bénignes, Détail |

#### 2. Nettoyage
- Colonne supprimée : `code_patho` (doublon de `libelle_patho`). Les libellés DREES sont abrégés : « T.M. » = tumeur maligne, « T.B. » = tumeur bénigne.
- 🔗 La correspondance avec ameli (tableau en fin de notebook) se lit désormais avec les libellés : « Tumeur maligne du sein » = `02017`, « Tumeurs malignes colo-rectales » = `02011`, « T.M. trachée_bronches_poumon » = `02015`, « Tumeur maligne de la prostate » = `02019`, « Tumeurs malignes » = `02007`.

#### 3. Un DataFrame par niveau de granularité
- Découpage avec la fonction `decouper_par_niveau`, et les noms des **référentiels ameli** à la place des codes (mêmes noms que dans le DF 1).

| DataFrame | Lignes | Colonnes géographiques | DMS manquantes |
|---|---:|---|---:|
| `df_dms_france` | 378 | `territoire` (France / France métropolitaine) | 0 |
| `df_dms_region` | 3 402 | `nom_region` | 6 |
| `df_dms_dept` | 19 089 | `nom_region`, `dept`, `nom_dept` | 47 |

Contrôle : 378 + 3 402 + 19 089 = 22 869 lignes, et aucun territoire sans nom.

#### 4. Points d'attention pour la suite
- **`df_dms_france` contient deux territoires** (France entière et métropole) : filtrer sur `territoire`.
- **Codes emboîtés** : utiliser `niveau_patho` pour ne pas mélanger « Tumeurs », « Tumeurs malignes » et les localisations détaillées.
- **DMS agrégée** : moyenne **pondérée** par `nb_sejours_24h_plus`, jamais une moyenne simple.
- La DMS est calculée **hors séjours de moins de 24 h** : à lire avec la part d'ambulatoire (DF 4).

In [62]:
display(df_dms_onco.head(10))

,annee,niveau,code_zone,libelle_zone,code_patho,libelle_patho,niveau_patho,nb_sejours_24h_plus,dms_jours
0,2018,Département,01,Ain,02000,Tumeurs,Chapitre (toutes tumeurs),6344.0,7.08
1,2018,Département,01,Ain,02007,Tumeurs malignes,Sous-total tumeurs malignes,4522.0,8.35
2,2018,Département,01,Ain,02008,T.M. lèvre _bouche_pharynx,Détail,104.0,10.00
3,2018,Département,01,Ain,02009,Tumeur maligne de l'oesophage,Détail,50.0,12.16
4,2018,Département,01,Ain,02010,Tumeur maligne de l'estomac,Détail,69.0,10.99
5,2018,Département,01,Ain,02011,Tumeurs malignes colo-rectales,Détail,425.0,10.90
6,2018,Département,01,Ain,02012,T.M.foie et voies biliaires intrahép,Détail,173.0,7.86
7,2018,Département,01,Ain,02013,Tumeur maligne du pancréas,Détail,127.0,11.08
8,2018,Département,01,Ain,02014,Tumeur maligne du larynx,Détail,50.0,14.49
9,2018,Département,01,Ain,02015,T.M. trachée_bronches_poumon,Détail,411.0,10.35


In [63]:
df_dms_onco.isna().sum()

annee                   0
niveau                  0
code_zone               0
libelle_zone            0
code_patho              0
libelle_patho           0
niveau_patho            0
nb_sejours_24h_plus     4
dms_jours              53
dtype: int64

In [64]:
df_dms_onco.duplicated().sum()

np.int64(0)

In [65]:
df_dms_onco.info()

<class 'pandas.DataFrame'>
RangeIndex: 22869 entries, 0 to 22868
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   annee                22869 non-null  int64  
 1   niveau               22869 non-null  str    
 2   code_zone            22869 non-null  str    
 3   libelle_zone         22869 non-null  str    
 4   code_patho           22869 non-null  str    
 5   libelle_patho        22869 non-null  str    
 6   niveau_patho         22869 non-null  str    
 7   nb_sejours_24h_plus  22865 non-null  float64
 8   dms_jours            22816 non-null  float64
dtypes: float64(2), int64(1), str(6)
memory usage: 3.0 MB


In [66]:
# Valeurs uniques de chaque colonne (la liste n'est affichée que s'il y a 50 valeurs ou moins)
for col in df_dms_onco.columns:
    valeurs = df_dms_onco[col].unique()
    print (f"{col} : {len(valeurs)} valeurs uniques")
    if len(valeurs) <= 50:
        print(f'les valeurs uniques sont : {valeurs.tolist()}')

annee : 7 valeurs uniques
les valeurs uniques sont : [2018, 2019, 2020, 2021, 2022, 2023, 2024]
niveau : 3 valeurs uniques
les valeurs uniques sont : ['Département', 'France', 'Région']
code_zone : 103 valeurs uniques
libelle_zone : 117 valeurs uniques
code_patho : 27 valeurs uniques
les valeurs uniques sont : ['02000', '02007', '02008', '02009', '02010', '02011', '02012', '02013', '02014', '02015', '02016', '02017', '02018', '02019', '02020', '02021', '02022', '02023', '02024', '02025', '02026', '02027', '02028', '02029', '02030', '02031', '02032']
libelle_patho : 27 valeurs uniques
les valeurs uniques sont : ['Tumeurs', 'Tumeurs malignes', 'T.M. lèvre _bouche_pharynx', "Tumeur maligne de l'oesophage", "Tumeur maligne de l'estomac", 'Tumeurs malignes colo-rectales', 'T.M.foie et voies biliaires intrahép', 'Tumeur maligne du pancréas', 'Tumeur maligne du larynx', 'T.M. trachée_bronches_poumon', 'Mélanome malin de la peau', 'Tumeur maligne du sein', "Tumeur maligne de l'utérus", 'Tumeur

In [67]:
# Où sont les DMS manquantes ?
df_dms_onco[df_dms_onco['dms_jours'].isna()].groupby(['niveau', 'libelle_patho']).size()

niveau       libelle_patho              
Département  Mélanome malin de la peau       1
             Tumeur bénigne du sein          8
             Tumeur maligne de l'estomac     1
             Tumeur maligne du larynx        2
             Tumeur maligne du pancréas      1
             Tumeurs bénignes de la peau    34
Région       Mélanome malin de la peau       1
             Tumeur bénigne du sein          1
             Tumeur maligne du larynx        1
             Tumeur maligne du pancréas      1
             Tumeurs bénignes de la peau     2
dtype: int64

In [68]:
print("code_patho <-> libelle_patho :", correspondance_1_1(df_dms_onco, 'code_patho', 'libelle_patho'))

code_patho <-> libelle_patho : True


In [69]:
df_dms_onco.drop(columns=['code_patho'], inplace=True)

In [70]:
df_dms_onco.head(10)

,annee,niveau,code_zone,libelle_zone,libelle_patho,niveau_patho,nb_sejours_24h_plus,dms_jours
0,2018,Département,01,Ain,Tumeurs,Chapitre (toutes tumeurs),6344.0,7.08
1,2018,Département,01,Ain,Tumeurs malignes,Sous-total tumeurs malignes,4522.0,8.35
2,2018,Département,01,Ain,T.M. lèvre _bouche_pharynx,Détail,104.0,10.00
3,2018,Département,01,Ain,Tumeur maligne de l'oesophage,Détail,50.0,12.16
4,2018,Département,01,Ain,Tumeur maligne de l'estomac,Détail,69.0,10.99
5,2018,Département,01,Ain,Tumeurs malignes colo-rectales,Détail,425.0,10.90
6,2018,Département,01,Ain,T.M.foie et voies biliaires intrahép,Détail,173.0,7.86
7,2018,Département,01,Ain,Tumeur maligne du pancréas,Détail,127.0,11.08
8,2018,Département,01,Ain,Tumeur maligne du larynx,Détail,50.0,14.49
9,2018,Département,01,Ain,T.M. trachée_bronches_poumon,Détail,411.0,10.35


### DF 3 découpé par niveau de granularité

Même démarche que pour le DF 1 : un DataFrame par échelle géographique, et les codes remplacés par des noms.
Les noms viennent des **référentiels ameli** (`nom_region`, `nom_dept`, chargés dans la partie DF 1), et non des libellés DREES. Ainsi, un même territoire porte exactement le même nom dans tous les DF (« Île-de-France » chez ameli contre « Ile de France » chez la DREES), ce qui permet de les joindre sur le nom.

La fonction `decouper_par_niveau` ci‑dessous est réutilisée pour le DF 4.

| DataFrame | Contenu | Colonnes géographiques | Lignes |
|---|---|---|---:|
| `df_dms_france` | France entière **et** France métropolitaine | `territoire` | 378 |
| `df_dms_region` | 18 régions | `nom_region` | 3 402 |
| `df_dms_dept` | 101 départements | `nom_region`, `dept`, `nom_dept` | 19 089 |

⚠️ `df_dms_france` contient deux territoires : filtrer sur `territoire` (« France » ou « France métropolitaine ») avant tout calcul.

In [71]:
# Découpage par niveau de granularité, avec les noms ameli à la place des codes (même logique que pour le DF 1)
region_du_dept = dict(zip(ref_depts['dep_code'], ref_depts['reg_code']))   # ex. '75' -> '11'

def decouper_par_niveau(df):
    colonnes_geo = ['niveau', 'code_zone', 'libelle_zone', 'code_region']
    colonnes_donnees = [c for c in df.columns if c not in colonnes_geo]

    france = df[df['niveau'] == 'France'].copy()
    france.insert(0, 'territoire', france['code_zone'].map({'FR': 'France', 'FRM': 'France métropolitaine'}))

    region = df[df['niveau'] == 'Région'].sort_values('code_zone', kind='stable').copy()
    region.insert(0, 'nom_region', region['code_zone'].map(nom_region))

    dept = df[df['niveau'] == 'Département'].copy()
    dept['code_reg_tri'] = dept['code_zone'].map(region_du_dept)
    dept = dept.sort_values(['code_reg_tri', 'code_zone'], kind='stable')
    dept.insert(0, 'nom_region', dept['code_reg_tri'].map(nom_region))
    dept.insert(1, 'dept', dept['code_zone'])
    dept.insert(2, 'nom_dept', dept['code_zone'].map(nom_dept))

    return (france[['territoire'] + colonnes_donnees].reset_index(drop=True),
            region[['nom_region'] + colonnes_donnees].reset_index(drop=True),
            dept[['nom_region', 'dept', 'nom_dept'] + colonnes_donnees].reset_index(drop=True))

In [72]:
df_dms_france, df_dms_region, df_dms_dept = decouper_par_niveau(df_dms_onco)

for nom, d in [('df_dms_france', df_dms_france), ('df_dms_region', df_dms_region), ('df_dms_dept', df_dms_dept)]:
    print(f"{nom} : {d.shape[0]} lignes x {d.shape[1]} colonnes")
print(f"Total : {len(df_dms_france) + len(df_dms_region) + len(df_dms_dept)} lignes (df_dms_onco : {len(df_dms_onco)})")
print("Territoires sans nom :", df_dms_region['nom_region'].isna().sum()
      + df_dms_dept[['nom_region', 'nom_dept']].isna().sum().sum())

df_dms_france : 378 lignes x 6 colonnes
df_dms_region : 3402 lignes x 6 colonnes
df_dms_dept : 19089 lignes x 8 colonnes
Total : 22869 lignes (df_dms_onco : 22869)
Territoires sans nom : 0


In [73]:
df_dms_france.info()

<class 'pandas.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   territoire           378 non-null    str    
 1   annee                378 non-null    int64  
 2   libelle_patho        378 non-null    str    
 3   niveau_patho         378 non-null    str    
 4   nb_sejours_24h_plus  378 non-null    float64
 5   dms_jours            378 non-null    float64
dtypes: float64(2), int64(1), str(3)
memory usage: 36.1 KB


In [74]:
df_dms_region.info()

<class 'pandas.DataFrame'>
RangeIndex: 3402 entries, 0 to 3401
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nom_region           3402 non-null   str    
 1   annee                3402 non-null   int64  
 2   libelle_patho        3402 non-null   str    
 3   niveau_patho         3402 non-null   str    
 4   nb_sejours_24h_plus  3400 non-null   float64
 5   dms_jours            3396 non-null   float64
dtypes: float64(2), int64(1), str(3)
memory usage: 320.9 KB


In [75]:
df_dms_dept.info()

<class 'pandas.DataFrame'>
RangeIndex: 19089 entries, 0 to 19088
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nom_region           19089 non-null  str    
 1   dept                 19089 non-null  str    
 2   nom_dept             19089 non-null  str    
 3   annee                19089 non-null  int64  
 4   libelle_patho        19089 non-null  str    
 5   niveau_patho         19089 non-null  str    
 6   nb_sejours_24h_plus  19087 non-null  float64
 7   dms_jours            19042 non-null  float64
dtypes: float64(2), int64(1), str(5)
memory usage: 2.3 MB


## Exploration — DF 4 : `df_charge_onco`

Charge hospitalière en court séjour (DREES) par code tumeur et par territoire : nombre de séjours, répartition par durée, part d'ambulatoire et journées estimées, 2018‑2024.

### Notes d'exploration

#### 1. Premier diagnostic de `df_charge_onco`
- **22 869 lignes × 25 colonnes**, **aucun doublon**.
- **Valeurs manquantes** : 4 lignes ont **toutes** leurs colonnes de séjours vides (non diffusées par la DREES, mêmes lignes que dans le DF 3). `nb_journees_estimees` est vide sur 53 lignes, car elle dépend de la DMS (mêmes lignes que les DMS manquantes du DF 3).
- **Valeurs uniques** : mêmes constats que le DF 3 pour les colonnes géographiques et de pathologie (`code_patho` ↔ `libelle_patho` : **redondant**, vérifié). Les colonnes de séjours ont des milliers de valeurs différentes.
- **Cohérence interne** vérifiée sur les 22 865 lignes complètes :
  - séjours de moins de 24 h + séjours par durée = total des séjours, à **16 séjours près au maximum** (petit écart sur 12 241 lignes) ;
  - programmés + autres = séjours de moins de 24 h, à **4 séjours près au maximum** (5 767 lignes).
  - Ces écarts sont négligeables. Ils viennent probablement des arrondis ou de la protection des petits effectifs appliqués par la DREES (hypothèse, non documentée dans le fichier).

#### 2. Nettoyage
- Colonne supprimée : `code_patho` (doublon de `libelle_patho`), comme pour le DF 3.

#### 3. Un DataFrame par niveau de granularité

| DataFrame | Lignes | Colonnes géographiques |
|---|---:|---|
| `df_charge_france` | 378 | `territoire` (France / France métropolitaine) |
| `df_charge_region` | 3 402 | `nom_region` |
| `df_charge_dept` | 19 089 | `nom_region`, `dept`, `nom_dept` |

Contrôle : 378 + 3 402 + 19 089 = 22 869 lignes, et aucun territoire sans nom.

#### 4. Points d'attention pour la suite
- **Séjours ≠ patients** : un patient peut avoir plusieurs séjours. Les séances de chimiothérapie et de radiothérapie sont comptées dans les séjours de moins de 24 h.
- **Court séjour (MCO) uniquement** : ni SSR, ni HAD, ni psychiatrie.
- **Codes emboîtés** (`niveau_patho`) et **deux territoires dans `df_charge_france`**, comme pour le DF 3.
- **2020 atypique** (Covid) ; **2024 provisoire**.

In [76]:
display(df_charge_onco.head(10))

,annee,niveau,code_zone,libelle_zone,code_patho,libelle_patho,niveau_patho,nb_sejours,nb_sejours_moins_24h,sej_moins_24h_programmes,...,sej_5j,sej_6j,sej_7j,sej_8j,sej_9j,sej_10_19j,sej_20_29j,sej_30j_plus,part_ambulatoire,nb_journees_estimees
0,2018,Département,01,Ain,02000,Tumeurs,Chapitre (toutes tumeurs),12850.0,6506.0,6484.0,...,350.0,315.0,289.0,248.0,203.0,866.0,305.0,204.0,0.506304,44915.52
1,2018,Département,01,Ain,02007,Tumeurs malignes,Sous-total tumeurs malignes,6657.0,2135.0,2115.0,...,273.0,245.0,228.0,203.0,166.0,769.0,284.0,191.0,0.320715,37758.70
2,2018,Département,01,Ain,02008,T.M. lèvre _bouche_pharynx,Détail,144.0,40.0,39.0,...,7.0,5.0,0.0,3.0,0.0,21.0,6.0,8.0,0.277778,1040.00
3,2018,Département,01,Ain,02009,Tumeur maligne de l'oesophage,Détail,71.0,21.0,21.0,...,0.0,0.0,0.0,4.0,3.0,9.0,8.0,5.0,0.295775,608.00
4,2018,Département,01,Ain,02010,Tumeur maligne de l'estomac,Détail,97.0,28.0,28.0,...,4.0,0.0,3.0,5.0,0.0,25.0,3.0,4.0,0.288660,758.31
5,2018,Département,01,Ain,02011,Tumeurs malignes colo-rectales,Détail,613.0,188.0,188.0,...,29.0,25.0,25.0,32.0,19.0,119.0,41.0,22.0,0.306688,4632.50
6,2018,Département,01,Ain,02012,T.M.foie et voies biliaires intrahép,Détail,182.0,9.0,6.0,...,19.0,10.0,11.0,0.0,10.0,33.0,11.0,3.0,0.049451,1359.78
7,2018,Département,01,Ain,02013,Tumeur maligne du pancréas,Détail,143.0,16.0,13.0,...,0.0,10.0,7.0,5.0,3.0,33.0,16.0,8.0,0.111888,1407.16
8,2018,Département,01,Ain,02014,Tumeur maligne du larynx,Détail,75.0,25.0,24.0,...,0.0,0.0,0.0,3.0,0.0,8.0,10.0,7.0,0.333333,724.50
9,2018,Département,01,Ain,02015,T.M. trachée_bronches_poumon,Détail,458.0,47.0,47.0,...,26.0,23.0,23.0,21.0,19.0,96.0,38.0,22.0,0.102620,4253.85


In [77]:
df_charge_onco.isna().sum()

annee                        0
niveau                       0
code_zone                    0
libelle_zone                 0
code_patho                   0
libelle_patho                0
niveau_patho                 0
nb_sejours                   4
nb_sejours_moins_24h         4
sej_moins_24h_programmes     4
sej_moins_24h_autres         4
sej_1j                       4
sej_2j                       4
sej_3j                       4
sej_4j                       4
sej_5j                       4
sej_6j                       4
sej_7j                       4
sej_8j                       4
sej_9j                       4
sej_10_19j                   4
sej_20_29j                   4
sej_30j_plus                 4
part_ambulatoire             4
nb_journees_estimees        53
dtype: int64

In [78]:
df_charge_onco.duplicated().sum()

np.int64(0)

In [79]:
df_charge_onco.info()

<class 'pandas.DataFrame'>
RangeIndex: 22869 entries, 0 to 22868
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   annee                     22869 non-null  int64  
 1   niveau                    22869 non-null  str    
 2   code_zone                 22869 non-null  str    
 3   libelle_zone              22869 non-null  str    
 4   code_patho                22869 non-null  str    
 5   libelle_patho             22869 non-null  str    
 6   niveau_patho              22869 non-null  str    
 7   nb_sejours                22865 non-null  float64
 8   nb_sejours_moins_24h      22865 non-null  float64
 9   sej_moins_24h_programmes  22865 non-null  float64
 10  sej_moins_24h_autres      22865 non-null  float64
 11  sej_1j                    22865 non-null  float64
 12  sej_2j                    22865 non-null  float64
 13  sej_3j                    22865 non-null  float64
 14  sej_4j           

In [80]:
# Valeurs uniques de chaque colonne (la liste n'est affichée que s'il y a 50 valeurs ou moins)
for col in df_charge_onco.columns:
    valeurs = df_charge_onco[col].unique()
    print (f"{col} : {len(valeurs)} valeurs uniques")
    if len(valeurs) <= 50:
        print(f'les valeurs uniques sont : {valeurs.tolist()}')

annee : 7 valeurs uniques
les valeurs uniques sont : [2018, 2019, 2020, 2021, 2022, 2023, 2024]
niveau : 3 valeurs uniques
les valeurs uniques sont : ['Département', 'France', 'Région']
code_zone : 103 valeurs uniques
libelle_zone : 117 valeurs uniques
code_patho : 27 valeurs uniques
les valeurs uniques sont : ['02000', '02007', '02008', '02009', '02010', '02011', '02012', '02013', '02014', '02015', '02016', '02017', '02018', '02019', '02020', '02021', '02022', '02023', '02024', '02025', '02026', '02027', '02028', '02029', '02030', '02031', '02032']
libelle_patho : 27 valeurs uniques
les valeurs uniques sont : ['Tumeurs', 'Tumeurs malignes', 'T.M. lèvre _bouche_pharynx', "Tumeur maligne de l'oesophage", "Tumeur maligne de l'estomac", 'Tumeurs malignes colo-rectales', 'T.M.foie et voies biliaires intrahép', 'Tumeur maligne du pancréas', 'Tumeur maligne du larynx', 'T.M. trachée_bronches_poumon', 'Mélanome malin de la peau', 'Tumeur maligne du sein', "Tumeur maligne de l'utérus", 'Tumeur

In [81]:
# Cohérence interne (lignes sans valeur manquante) :
# séjours < 24 h + séjours par durée = total ; programmés + autres = séjours < 24 h
complet = df_charge_onco.dropna(subset=['nb_sejours', 'nb_sejours_moins_24h', 'sej_moins_24h_programmes',
                                        'sej_moins_24h_autres'] + colonnes_duree)
ecart_total = (complet['nb_sejours_moins_24h'] + complet[colonnes_duree].sum(axis=1) - complet['nb_sejours']).abs()
ecart_24h = (complet['sej_moins_24h_programmes'] + complet['sej_moins_24h_autres'] - complet['nb_sejours_moins_24h']).abs()
print(f"Lignes complètes : {len(complet)} / {len(df_charge_onco)}")
print(f"Écart max total : {ecart_total.max():.0f} séjours ({(ecart_total > 0).sum()} lignes avec un écart)")
print(f"Écart max < 24 h : {ecart_24h.max():.0f} séjours ({(ecart_24h > 0).sum()} lignes avec un écart)")

Lignes complètes : 22865 / 22869
Écart max total : 16 séjours (12241 lignes avec un écart)
Écart max < 24 h : 4 séjours (5767 lignes avec un écart)


In [82]:
print("code_patho <-> libelle_patho :", correspondance_1_1(df_charge_onco, 'code_patho', 'libelle_patho'))

code_patho <-> libelle_patho : True


In [83]:
df_charge_onco.drop(columns=['code_patho'], inplace=True)

In [84]:
df_charge_onco.head(10)

,annee,niveau,code_zone,libelle_zone,libelle_patho,niveau_patho,nb_sejours,nb_sejours_moins_24h,sej_moins_24h_programmes,sej_moins_24h_autres,...,sej_5j,sej_6j,sej_7j,sej_8j,sej_9j,sej_10_19j,sej_20_29j,sej_30j_plus,part_ambulatoire,nb_journees_estimees
0,2018,Département,01,Ain,Tumeurs,Chapitre (toutes tumeurs),12850.0,6506.0,6484.0,22.0,...,350.0,315.0,289.0,248.0,203.0,866.0,305.0,204.0,0.506304,44915.52
1,2018,Département,01,Ain,Tumeurs malignes,Sous-total tumeurs malignes,6657.0,2135.0,2115.0,20.0,...,273.0,245.0,228.0,203.0,166.0,769.0,284.0,191.0,0.320715,37758.70
2,2018,Département,01,Ain,T.M. lèvre _bouche_pharynx,Détail,144.0,40.0,39.0,0.0,...,7.0,5.0,0.0,3.0,0.0,21.0,6.0,8.0,0.277778,1040.00
3,2018,Département,01,Ain,Tumeur maligne de l'oesophage,Détail,71.0,21.0,21.0,0.0,...,0.0,0.0,0.0,4.0,3.0,9.0,8.0,5.0,0.295775,608.00
4,2018,Département,01,Ain,Tumeur maligne de l'estomac,Détail,97.0,28.0,28.0,0.0,...,4.0,0.0,3.0,5.0,0.0,25.0,3.0,4.0,0.288660,758.31
5,2018,Département,01,Ain,Tumeurs malignes colo-rectales,Détail,613.0,188.0,188.0,0.0,...,29.0,25.0,25.0,32.0,19.0,119.0,41.0,22.0,0.306688,4632.50
6,2018,Département,01,Ain,T.M.foie et voies biliaires intrahép,Détail,182.0,9.0,6.0,3.0,...,19.0,10.0,11.0,0.0,10.0,33.0,11.0,3.0,0.049451,1359.78
7,2018,Département,01,Ain,Tumeur maligne du pancréas,Détail,143.0,16.0,13.0,3.0,...,0.0,10.0,7.0,5.0,3.0,33.0,16.0,8.0,0.111888,1407.16
8,2018,Département,01,Ain,Tumeur maligne du larynx,Détail,75.0,25.0,24.0,0.0,...,0.0,0.0,0.0,3.0,0.0,8.0,10.0,7.0,0.333333,724.50
9,2018,Département,01,Ain,T.M. trachée_bronches_poumon,Détail,458.0,47.0,47.0,0.0,...,26.0,23.0,23.0,21.0,19.0,96.0,38.0,22.0,0.102620,4253.85


### DF 4 découpé par niveau de granularité

Même découpage que pour le DF 3, avec la fonction `decouper_par_niveau` et les noms des référentiels ameli.

| DataFrame | Contenu | Colonnes géographiques | Lignes |
|---|---|---|---:|
| `df_charge_france` | France entière **et** France métropolitaine | `territoire` | 378 |
| `df_charge_region` | 18 régions | `nom_region` | 3 402 |
| `df_charge_dept` | 101 départements | `nom_region`, `dept`, `nom_dept` | 19 089 |

⚠️ `df_charge_france` contient deux territoires : filtrer sur `territoire` avant tout calcul.

In [85]:
df_charge_france, df_charge_region, df_charge_dept = decouper_par_niveau(df_charge_onco)

for nom, d in [('df_charge_france', df_charge_france), ('df_charge_region', df_charge_region), ('df_charge_dept', df_charge_dept)]:
    print(f"{nom} : {d.shape[0]} lignes x {d.shape[1]} colonnes")
print(f"Total : {len(df_charge_france) + len(df_charge_region) + len(df_charge_dept)} lignes (df_charge_onco : {len(df_charge_onco)})")
print("Territoires sans nom :", df_charge_region['nom_region'].isna().sum()
      + df_charge_dept[['nom_region', 'nom_dept']].isna().sum().sum())

df_charge_france : 378 lignes x 22 colonnes
df_charge_region : 3402 lignes x 22 colonnes
df_charge_dept : 19089 lignes x 24 colonnes
Total : 22869 lignes (df_charge_onco : 22869)
Territoires sans nom : 0


In [86]:
df_charge_france.info()

<class 'pandas.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   territoire                378 non-null    str    
 1   annee                     378 non-null    int64  
 2   libelle_patho             378 non-null    str    
 3   niveau_patho              378 non-null    str    
 4   nb_sejours                378 non-null    float64
 5   nb_sejours_moins_24h      378 non-null    float64
 6   sej_moins_24h_programmes  378 non-null    float64
 7   sej_moins_24h_autres      378 non-null    float64
 8   sej_1j                    378 non-null    float64
 9   sej_2j                    378 non-null    float64
 10  sej_3j                    378 non-null    float64
 11  sej_4j                    378 non-null    float64
 12  sej_5j                    378 non-null    float64
 13  sej_6j                    378 non-null    float64
 14  sej_7j               

In [87]:
df_charge_region.info()

<class 'pandas.DataFrame'>
RangeIndex: 3402 entries, 0 to 3401
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   nom_region                3402 non-null   str    
 1   annee                     3402 non-null   int64  
 2   libelle_patho             3402 non-null   str    
 3   niveau_patho              3402 non-null   str    
 4   nb_sejours                3400 non-null   float64
 5   nb_sejours_moins_24h      3400 non-null   float64
 6   sej_moins_24h_programmes  3400 non-null   float64
 7   sej_moins_24h_autres      3400 non-null   float64
 8   sej_1j                    3400 non-null   float64
 9   sej_2j                    3400 non-null   float64
 10  sej_3j                    3400 non-null   float64
 11  sej_4j                    3400 non-null   float64
 12  sej_5j                    3400 non-null   float64
 13  sej_6j                    3400 non-null   float64
 14  sej_7j             

In [88]:
df_charge_dept.info()

<class 'pandas.DataFrame'>
RangeIndex: 19089 entries, 0 to 19088
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   nom_region                19089 non-null  str    
 1   dept                      19089 non-null  str    
 2   nom_dept                  19089 non-null  str    
 3   annee                     19089 non-null  int64  
 4   libelle_patho             19089 non-null  str    
 5   niveau_patho              19089 non-null  str    
 6   nb_sejours                19087 non-null  float64
 7   nb_sejours_moins_24h      19087 non-null  float64
 8   sej_moins_24h_programmes  19087 non-null  float64
 9   sej_moins_24h_autres      19087 non-null  float64
 10  sej_1j                    19087 non-null  float64
 11  sej_2j                    19087 non-null  float64
 12  sej_3j                    19087 non-null  float64
 13  sej_4j                    19087 non-null  float64
 14  sej_5j           

# Export CSV des DataFrames finaux

Les 13 DataFrames obtenus après exploration sont enregistrés dans le **dossier de travail** (celui du notebook), un fichier `.csv` par DataFrame.

| Jeu | Table complète | Découpage France / Région / Département |
|---|---|---|
| DF 1 — patients cancer du sein | `df_patients_sein` | `df_sein_france`, `df_sein_region`, `df_sein_dept` |
| DF 2 — dépenses oncologie | `df_depenses_onco` | *(France uniquement, pas de découpage)* |
| DF 3 — DMS oncologie | `df_dms_onco` | `df_dms_france`, `df_dms_region`, `df_dms_dept` |
| DF 4 — charge hospitalière | `df_charge_onco` | `df_charge_france`, `df_charge_region`, `df_charge_dept` |

**Format « Excel français »** : séparateur `;`, virgule décimale, encodage UTF‑8 avec BOM (les accents s'affichent correctement dans Excel).
Pour relire un fichier avec pandas : `pd.read_csv('df_sein_dept.csv', sep=';', decimal=',', encoding='utf-8-sig', dtype={'dept': str})`.

In [ ]:
# Export CSV de tous les DataFrames finaux, dans le dossier de travail (dossier du notebook)
EXPORT_DIR = Path('.')

dataframes_a_exporter = {
    # DF 1 — patients atteints d'un cancer du sein
    'df_patients_sein': df_patients_sein,
    'df_sein_france': df_sein_france,
    'df_sein_region': df_sein_region,
    'df_sein_dept': df_sein_dept,
    # DF 2 — dépenses remboursées (France)
    'df_depenses_onco': df_depenses_onco,
    # DF 3 — durée moyenne de séjour
    'df_dms_onco': df_dms_onco,
    'df_dms_france': df_dms_france,
    'df_dms_region': df_dms_region,
    'df_dms_dept': df_dms_dept,
    # DF 4 — charge hospitalière
    'df_charge_onco': df_charge_onco,
    'df_charge_france': df_charge_france,
    'df_charge_region': df_charge_region,
    'df_charge_dept': df_charge_dept,
}

for nom, df in dataframes_a_exporter.items():
    chemin = EXPORT_DIR / f'{nom}.csv'
    df.to_csv(chemin, sep=';', decimal=',', index=False, encoding='utf-8-sig')
    # Contrôle : le fichier relu a bien les mêmes dimensions que le DataFrame
    relu = pd.read_csv(chemin, sep=';', decimal=',', encoding='utf-8-sig', low_memory=False)
    statut = 'OK' if relu.shape == df.shape else f'ÉCART (relu : {relu.shape})'
    print(f"{chemin.name:<24} {df.shape[0]:>7} lignes x {df.shape[1]:>2} colonnes  "
          f"{chemin.stat().st_size / 1e6:6.2f} Mo  {statut}")

print()
print(f"{len(dataframes_a_exporter)} fichiers exportés dans : {EXPORT_DIR.resolve()}")